# Chapter 6 — Neuroimaging and Functional Connectivity

This exercise uses **synthetic, non-clinical** NIfTI data. It demonstrates voxel geometry, affine transformations, anatomical slices, regional time courses, and correlation-based functional connectivity. It does **not** simulate realistic BOLD physiology or diagnose any condition.


In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))
from exercise_6_1_neuroimaging import generate_demo, analyze
data_dir = generate_demo(Path('outputs/ch06/synthetic_data'))
correlation = analyze(data_dir, Path('outputs/ch06'))
print('ROI connectivity matrix:', correlation.round(3))


## Inspect geometry and outputs
The affine maps voxel indices `(i,j,k)` to physical coordinates in millimeters. These synthetic volumes share a grid by construction; real acquisitions must be checked and registered before atlas extraction.


In [ ]:
import json
import nibabel as nib
import numpy as np
from IPython.display import display, Image
out = Path('outputs/ch06')
t1 = nib.load(out / 'synthetic_data' / 'synthetic_T1w.nii.gz')
fmri = nib.load(out / 'synthetic_data' / 'synthetic_bold.nii.gz')
summary = json.loads((out / 'analysis_summary.json').read_text())
print('Anatomical shape:', t1.shape)
print('fMRI shape:', fmri.shape)
print('Voxel spacing (mm):', t1.header.get_zooms()[:3])
print('TR (seconds):', fmri.header.get_zooms()[3])
print('Affine (voxel index to mm):\n', t1.affine)
print('Voxel (16,16,12) maps to mm:', nib.affines.apply_affine(t1.affine, [16,16,12]))
print('ROI correlation matrix:\n', np.round(correlation, 3))
display(Image(filename=str(out / 'anatomical_slices.png')))
display(Image(filename=str(out / 'connectivity.png')))


## Explore and interpret
1. Open `outputs/ch06/analysis_summary.json`. Explain voxel spacing, the affine matrix, TR, and image dimensions.
2. Examine `anatomical_slices.png` and explain why the displayed axes are not physical anatomical coordinates unless the affine and orientation are checked.
3. Examine `connectivity.png`. Explain why correlation is not anatomical or causal connectivity.
4. Graduate extension: compare Pearson correlations before and after a documented time-series transformation, discuss how analytic flexibility changes interpretation, and avoid fitting or claiming a clinical model.
5. Optional public-data extension: use an appropriately licensed, already-preprocessed BIDS dataset with matching atlas and anatomical registration. Document acquisition, preprocessing, confounds, TR, atlas version, spatial transforms, and data access terms. Do **not** pass raw fMRI directly into this synthetic-data teaching script or assume identical voxel grids.


## Graduate quantitative sensitivity exercise
The optional transformation below removes a fitted linear trend from each ROI time series. Compare the matrices and describe how preprocessing choices change the estimated statistical association. This is not a validated nuisance-regression strategy for real fMRI data.


In [ ]:
from scipy.signal import detrend
roi = np.loadtxt(Path('outputs/ch06/roi_timeseries.csv'), delimiter=',', skiprows=1)
detrended = detrend(roi, axis=0, type='linear')
corr_detrended = np.corrcoef(detrended, rowvar=False)
print('Raw correlation:\n', np.round(correlation, 3))
print('Detrended correlation:\n', np.round(corr_detrended, 3))
print('Largest absolute matrix difference:', round(float(np.max(np.abs(corr_detrended - correlation))), 4))
